In [7]:
import numpy as np

_R = 8.314  # J/mol/K

def get_property(property, temperature, material_dict):
    """
    Calculate hydrogen transport properties using the Arrhenius equation.

    Parameters
    ----------
    property : str
        Property to compute: 'diffusivity', 'solubility', or 'permeability'
    temperature : float
        Absolute temperature [K]
    material_dict : dict
        Material parameters. Required keys per property:
        - 'diffusivity'  : 'D_0' [m²/s],          'E_D' [J/mol]
        - 'solubility'   : 'K_s0' [mol/m³/Pa⁰·⁵], 'H_s' [J/mol]
        - 'permeability' : all four keys above

    Returns
    -------
    float
        diffusivity  [m²/s]
        solubility   [mol/m³/Pa⁰·⁵]
        permeability [mol/m/s/Pa⁰·⁵]

    Raises
    ------
    ValueError
        If temperature ≤ 0 or property name is unrecognised.

    Notes
    -----
    Arrhenius form:  X = X₀ · exp(−Eₐ / RT)
    Permeability:    P = D · Ks
    """
    if temperature <= 0:
        raise ValueError(f"Temperature must be positive, got {temperature} K")

    def _arrhenius(pre_exp, activation_energy):
        return pre_exp * np.exp(-activation_energy / (_R * temperature))

    if property == 'diffusivity':
        return _arrhenius(material_dict['D_0'], material_dict['E_D'])
    elif property == 'solubility':
        return _arrhenius(material_dict['K_s0'], material_dict['H_s'])
    elif property == 'permeability':
        D   = _arrhenius(material_dict['D_0'],  material_dict['E_D'])
        K_s = _arrhenius(material_dict['K_s0'], material_dict['H_s'])
        return D * K_s
    else:
        raise ValueError(
            f"Unknown property '{property}'. "
            "Choose 'diffusivity', 'solubility', or 'permeability'."
        )

In [ ]:
props = ['diffusivity', 'solubility', 'permeability']
temp = np.linspace(1020, 1220, 5)  # K
material_dict = {
    'D_0': 3.4e-8,        # m²/s
    'E_D': 35400,        # J/mol
    'K_s0': 1.18,        # mol/m³/Pa^0.5
    'H_s': 26890,       # J/mol
}
results = {prop: [get_property(prop, T, material_dict) for T in temp] for prop in props}

import pandas as pd

print("="*60)
print("Name of Material and paper reference")
print("="*60)

df = pd.DataFrame(results, index=temp)

df.index.name = 'Temperature (K)'

df

Name of Material and paper reference


,diffusivity,solubility,permeability
Temperature (K),,,
5.230e+02,2.514e-09,2.433e-03,6.116e-12
6.105e+02,4.186e-09,5.903e-03,2.471e-11
6.980e+02,6.134e-09,1.147e-02,7.035e-11
7.855e+02,8.255e-09,1.922e-02,1.586e-10
8.730e+02,1.047e-08,2.903e-02,3.039e-10


In [ ]:
import numpy as np
import pandas as pd

# =============================================================================
# Unit Conversion Table (Table A1)
# All factors convert TO base SI units used internally:
#   Diffusivity  : m² s⁻¹
#   Solubility   : mol m⁻³ Pa⁻⁰·⁵
#   Permeability : mol m⁻¹ s⁻¹ Pa⁻⁰·⁵
#   Energy       : J mol⁻¹
#   Pressure     : Pa
#
# To convert FROM a unit TO SI  → multiply by the factor
# To convert FROM SI TO a unit  → divide by the factor
# =============================================================================

TO_SI = {
    # --- Diffusivity ---
    'm2/s'              : 1.0,
    'cm2/s'             : 1e-4,                # cm² s⁻¹  → m² s⁻¹

    # --- Solubility ---
    'mol/m3/Pa0.5'      : 1.0,
    'mol/m3/MPa0.5'     : 1e-3,                # MPa⁰·⁵ = 10³ Pa⁰·⁵
    'mol/m3/atm0.5'     : 3.14153e-3,          # ×3.14153 → mol/m³/MPa⁰·⁵, then ×1e-3
    'cm3/cm3/atm0.5'    : 140.16e-3,           # ×140.16  → mol/m³/MPa⁰·⁵, then ×1e-3
    'wtppm_H/MPa0.5'    : 3.9e-3,              # ×3.9     → mol/m³/MPa⁰·⁵, then ×1e-3
    'atppm_H/MPa0.5'    : 0.0718e-3,           # ×0.0718  → mol/m³/MPa⁰·⁵, then ×1e-3

    # --- Permeability ---
    'mol/m/s/Pa0.5'     : 1.0,
    'mol/m/s/MPa0.5'    : 1e-3,
    'mol/m/s/atm0.5'    : 3.14153e-3,          # ×3.14153 → mol/m/s/MPa⁰·⁵, then ×1e-3
    'cm3/cm/s/atm0.5'   : 0.014016e-3,         # ×0.014016→ mol/m/s/MPa⁰·⁵, then ×1e-3

    # --- Energy ---
    'J/mol'             : 1.0,
    'kJ/mol'            : 1e3,
    'eV'                : 96486.0,             # eV → kJ/mol (×96.486) → J/mol (×1e3)
    'cal/mol'           : 4.184,
    'kcal/mol'          : 4184.0,

    # --- Pressure ---
    'Pa'                : 1.0,
    'MPa'               : 1e6,
    'atm'               : 101325.0,
}

def convert(value, from_unit, to_unit='SI'):
    """
    Convert a value between units using Table A1 factors.

    Parameters
    ----------
    value : float or array-like
    from_unit : str   Key from TO_SI (e.g. 'cm2/s', 'eV', 'kJ/mol')
    to_unit   : str   Key from TO_SI, or 'SI' for base SI unit (default)

    Returns
    -------
    float or array-like

    Examples
    --------
    >>> convert(1e-3, 'cm2/s')                  # → m²/s
    >>> convert(0.044, 'eV')                    # → J/mol
    >>> convert(results_si, 'SI', 'cm2/s')      # SI → cm²/s
    """
    for unit in (from_unit, to_unit):
        if unit != 'SI' and unit not in TO_SI:
            raise ValueError(f"Unknown unit '{unit}'.\nAvailable: {list(TO_SI.keys())}")

    si_value = value * (TO_SI[from_unit] if from_unit != 'SI' else 1.0)

    if to_unit == 'SI':
        return si_value
    return si_value / TO_SI[to_unit]


# =============================================================================
# Material parameters — specify units explicitly, store in SI
# =============================================================================

material_dict = {
    'D_0': convert(1e-7,  'm2/s'),      # pre-exponential diffusivity
    'E_D': convert(4200,  'J/mol'),     # diffusion activation energy
    'K_s0': convert(0.1,  'mol/m3/Pa0.5'),  # pre-exponential solubility
    'H_s': convert(28600, 'J/mol'),     # solubility enthalpy
}

# =============================================================================
# Compute properties
# =============================================================================

props = ['diffusivity', 'solubility', 'permeability']
temp  = np.linspace(300, 1000, 5)  # K

results_si = {
    prop: [get_property(prop, T, material_dict) for T in temp]
    for prop in props
}

# =============================================================================
# Display in SI and common literature units
# =============================================================================

# ---- SI base units ----
df_si = pd.DataFrame(results_si, index=temp)
df_si.index.name = 'Temperature (K)'
df_si.columns = [
    'Diffusivity (m² s⁻¹)',
    'Solubility (mol m⁻³ Pa⁻⁰·⁵)',
    'Permeability (mol m⁻¹ s⁻¹ Pa⁻⁰·⁵)',
]

# ---- Common literature units ----
DISPLAY_UNITS = {
    'diffusivity'  : ('cm2/s',          'Diffusivity (cm² s⁻¹)'),
    'solubility'   : ('mol/m3/MPa0.5',  'Solubility (mol m⁻³ MPa⁻⁰·⁵)'),
    'permeability' : ('mol/m/s/MPa0.5', 'Permeability (mol m⁻¹ s⁻¹ MPa⁻⁰·⁵)'),
}

df_lit = pd.DataFrame({
    label: convert(np.array(results_si[prop]), 'SI', unit)
    for prop, (unit, label) in DISPLAY_UNITS.items()
}, index=temp)
df_lit.index.name = 'Temperature (K)'

# =============================================================================
# Print results
# =============================================================================

pd.set_option('display.float_format', '{:.3e}'.format)

header = "Name of Material and paper reference"
print("=" * 60)
print(header)
print("=" * 60)

print("\n[SI Units]")
print(df_si.to_string())

print("\n[Literature Units]")
print(df_lit.to_string())

Name of Material and paper reference

[SI Units]
                 Diffusivity (m² s⁻¹)  Solubility (mol m⁻³ Pa⁻⁰·⁵)  Permeability (mol m⁻¹ s⁻¹ Pa⁻⁰·⁵)
Temperature (K)                                                                                      
3.000e+02                   1.856e-08                    1.047e-06                          1.944e-14
4.750e+02                   3.452e-08                    7.158e-05                          2.471e-12
6.500e+02                   4.597e-08                    5.030e-04                          2.312e-11
8.250e+02                   5.421e-08                    1.546e-03                          8.379e-11
1.000e+03                   6.034e-08                    3.207e-03                          1.935e-10

[Literature Units]
                 Diffusivity (cm² s⁻¹)  Solubility (mol m⁻³ MPa⁻⁰·⁵)  Permeability (mol m⁻¹ s⁻¹ MPa⁻⁰·⁵)
Temperature (K)                                                                                         
3.000e+